# Trial Eligibility

**Assess number of patients in the aUC analytic cohort meeting CheckMate 214 ECOG and lab eligibility.**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorRenal, merge_dataframes

## Import dfs

In [2]:
index_df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [3]:
index_df.shape

(4831, 3)

In [4]:
dtype_map = pd.read_csv('../outputs/ioio_tki_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/ioio_tki_features.csv', dtype = dtype_map)

In [5]:
features_df.shape

(4366, 161)

In [6]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [7]:
surv_pred_df.shape

(4193, 2)

In [8]:
main_df = pd.merge(features_df, index_df, on = 'PatientID', how = 'left')

In [9]:
main_df = pd.merge(main_df, surv_pred_df, on = 'PatientID', how = 'left')

In [10]:
main_df = main_df.query('met_diagnosis_year <= 2020')

In [11]:
main_df.shape

(4193, 164)

In [12]:
main_df = main_df[['PatientID', 'LineName', 'StartDate', 'psurv_180_calibrated']]

In [13]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated
0,F42ACFD609CE6,tki,2015-05-08,0.868056
1,F2A697A9BC9E5,tki,2016-11-02,0.554913
2,F1D409FF1E1DF,tki,2013-09-13,0.959998


## Labs

In [14]:
# Initialize class 
processor = DataProcessorRenal()

In [15]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-04-25 23:21:47,614 - INFO - Successfully read Lab.csv file with shape: (7341289, 17) and unique PatientIDs: 10244
2026-04-25 23:21:50,132 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (3464060, 18) and unique PatientIDs: 4061
2026-04-25 23:21:56,919 - INFO - Successfully processed Lab.csv file with final shape: (4193, 81) and unique PatientIDs: 4193


In [16]:
labs_df = labs_df[['PatientID', 'wbc', 'neutrophil', 'platelet', 'creatinine', 'total_bilirubin', 'ast', 'alt']]

In [17]:
labs_df.shape

(4193, 9)

In [18]:
main_df = pd.merge(main_df, labs_df, on = 'PatientID', how = 'left')

In [19]:
main_df.shape

(4193, 12)

In [20]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated,wbc,neutrophil,platelet,creatinine,total_bilirubin,ast,alt,alp
0,F42ACFD609CE6,tki,2015-05-08,0.868056,9.2,7.176,200.0,NaN,NaN,NaN,NaN,NaN
1,F2A697A9BC9E5,tki,2016-11-02,0.554913,8.2,6.100,234.0,1.13,NaN,NaN,NaN,NaN
2,F1D409FF1E1DF,tki,2013-09-13,0.959998,10.2,NaN,309.0,1.28,0.3,17.0,20.0,76.0


## Risk stratification

In [21]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate = float(f.read())

In [22]:
main_df['high_risk'] = np.where(main_df['psurv_180_calibrated'] < crossover_survival_estimate, 1, 0)

In [23]:
main_df.high_risk.value_counts()

high_risk
0    2795
1    1398
Name: count, dtype: int64

## Lab eligibility

In [24]:
main_df['lab_eligible'] = np.where(
    (main_df['wbc'] >= 2) &
    (main_df['neutrophil'] >= 1.5) &
    (main_df['platelet'] >= 100) &
    (main_df['creatinine'] <= 1.8) & 
    (main_df['total_bilirubin'] <= 1.8) &
    (main_df['ast'] <= 120) &
    (main_df['alt'] <= 120), 1, 0)

In [25]:
main_df['lab_ineligible'] = np.where(
    (main_df['wbc'] < 2) &
    (main_df['neutrophil'] < 1.5) |
    (main_df['platelet'] < 100) |
    (main_df['creatinine'] > 1.8) | 
    (main_df['total_bilirubin'] > 1.8) |
    (main_df['ast'] > 120) |
    (main_df['alt'] > 120), 1, 0)

In [26]:
main_df['lab_indeterminate'] = np.where((main_df['lab_eligible'] == 0) & (main_df['lab_ineligible'] == 0), 1, 0)

In [27]:
main_df.lab_eligible.value_counts()[1]+main_df.lab_ineligible.value_counts()[1]+main_df.lab_indeterminate.value_counts()[1]

np.int64(4193)

## Overal eligibility

In [28]:
print('Overall cohort')
print(f'Eligible: count {main_df.lab_eligible.value_counts()[1]}, percentage {round(main_df.lab_eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.lab_ineligible.value_counts()[1]}, percentage {round(main_df.lab_ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.lab_indeterminate.value_counts()[1]}, percentage {round(main_df.lab_indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('High-risk')
print(f'Eligible: count {main_df.query('high_risk == 1').lab_eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').lab_eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 1').lab_ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').lab_ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 1').lab_indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').lab_indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('Low-risk')
print(f'Eligible: count {main_df.query('high_risk == 0').lab_eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').lab_eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 0').lab_ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').lab_ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 0').lab_indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').lab_indeterminate.value_counts(normalize = True)[1]*100, 2)}')

Overall cohort
Eligible: count 1636, percentage 39.02
Ineligible: count 414, percentage 9.87
Indeterminate: count 2143, percentage 51.11

High-risk
Eligible: count 576, percentage 41.2
Ineligible: count 195, percentage 13.95
Indeterminate: count 627, percentage 44.85

Low-risk
Eligible: count 1060, percentage 37.92
Ineligible: count 219, percentage 7.84
Indeterminate: count 1516, percentage 54.24
